## Final Project: Analysis Notebook

This is the starting point for the *analysis notebook*. It loads the same
tools the course notebooks used all semester, then reads in the tables you
collected over your two weeks along with the provided weather table. Run the
setup cells, check that every table loaded and that the keys line up, and
write your findings below the last cell.

### Setup

Run all of the following before starting. These are the same setup lines
the course notebooks used.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
from pathlib import Path

import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

### Loading the data

The cell below opens a file chooser. Select **all** of your CSV files at
once: `hours`, `places`, `activities`, `courses`, and the table of your own.
You do not need to upload the weather table. They are copied onto the Colab
disk next to this notebook. The check at the top means a rerun of the cell in
the same session does not ask again. The disk is wiped when the runtime
disconnects, so a new session starts with the upload.

In [ ]:
from google.colab import files

if not Path("hours.csv").exists():
    uploaded = files.upload()

sorted(p.name for p in Path(".").glob("*.csv"))

The list above shows every CSV file that is now on the disk. The next cell
reads the four tables that every project has. If one of your files has a
different name, change the name inside the quotes to match the list.

In [ ]:
hours = pl.read_csv("hours.csv")
places = pl.read_csv("places.csv")
activities = pl.read_csv("activities.csv")
courses = pl.read_csv("courses.csv")

The `weather` table is read straight from the course website, so there is
nothing to upload. It always has all 336 hours of the collection window, but
the hours that have not been filled in yet have missing values. The complete
table is posted on November 16th, the morning after collection ends. Rerun
the cell after that date to get the rest of the weather. The values are in
metric units: `temp` and `dewp` in degrees Celsius, `humid` in percent,
`wind_speed` in kilometers per hour, `precip` in millimeters, and `visib` in
kilometers.

In [ ]:
weather = pl.read_csv(
    "https://raw.githubusercontent.com/taylor-arnold/courses/refs/heads/main/dsst289-f26/extra/weather.csv"
)

Your own table has a name that only you know. Read it in below, replacing
both `mytable` and the file name with yours, and add a line for each further
table you collected.

In [ ]:
mytable = pl.read_csv("mytable.csv")

In [ ]:
for name, df in [("hours", hours), ("places", places), ("activities", activities),
                 ("courses", courses), ("weather", weather), ("mytable", mytable)]:
    print(f"{name:12s} {df.shape[0]:>5,} rows  {df.shape[1]:>3} columns")

A complete `hours` table has 336 rows, and so does `weather`.

### Checking the keys

The checks below are worth running before any finding, so that a key that
does not resolve shows up now and not in the middle of a plot. First, `date`
and `hour` together should identify each row of `hours`. The number of
distinct pairs should equal the number of rows.

In [ ]:
print(hours.shape[0], "rows")
print(hours.select(c.date, c.hour).unique().shape[0], "distinct date and hour pairs")

Second, the weather join. Every hour of the collection window is in
`weather`, so an inner join should not change the number of rows in `hours`.
If it does, look at how `date` and `hour` are written in the two tables. They
need the same type and the same format, such as `2026-11-02` and `0` through
`23`.

In [ ]:
hours_weather = hours.join(weather, on=["date", "hour"])
print(hours.shape[0], "rows in hours")
print(hours_weather.shape[0], "rows after joining weather")

In [ ]:
hours.join(weather, on=["date", "hour"], how="anti")

The anti join above shows the rows of `hours` that found no match in
`weather`. It should be empty.

Third, your own keys. Every place, activity, and course that appears in
`hours` should have a row in its own table. The key columns have names that
you chose, so the lines below are commented out. Remove the `#`, replace each
column name with yours, and run the cell. Each anti join should come back
empty, except that hours with no course are expected to have a missing course
key. Use `left_on=` and `right_on=` if a key has a different name in the two
tables.

In [ ]:
# hours.join(places, on="place_id", how="anti")
# hours.join(activities, on="activity_id", how="anti")
# hours.filter(c.course_id.is_not_null()).join(courses, on="course_id", how="anti")

### A look at each table

In [ ]:
hours

In [ ]:
places

In [ ]:
activities

In [ ]:
courses

In [ ]:
weather

In [ ]:
mytable

### Findings

Three to five findings that answer your research questions. Each finding
needs the question, the code, and a sentence or two saying what the result
shows. A finding that the data cannot settle a question is a real finding, as
long as you say what you would have needed to collect instead.

Across the notebook you need each of the following. Label each one in a text
cell where it appears, for example "**Join**" on the line above the code.

- at least one join
- at least one `.group_by` with `.agg`
- at least one window function with `.over`
- at least one pivot or unpivot
- at least four plots using at least three different geometries, of which at
  least one is faceted and at least one is fully labeled with `labs`

See the project description for the full requirements. When you are done,
hand the notebook in as a `.py` file (File → Download → Download .py).

#### Finding 1